# Preparar dataset de muros para fine-tuning propio de MLSTRUCT-FP

Toma los pares (PNG limpio, `muros_geo` corregido por el arquitecto) exportados desde el portal ArchiCheck (botón "📦 Exportar dataset de muros", habilitado tras confirmar la revisión gráfica) y genera los pares de entrenamiento reales: imagen 256×256 en escala de grises + máscara binaria de muros a la misma resolución.

**Qué NO hace este notebook**: no entrena nada. Es solo preparación de dataset — el fine-tuning en GPU es un paso futuro, después de acumular suficientes planos validados (ver roadmap, ítem P1b).

**Contexto de las unidades** (verificado contra el código real de extracción, no supuesto): `ancho_linea_prom` de cada muro viene en **puntos PDF** (`path.get('width')` de PyMuPDF, sin pasar por el `to_px()` que sí se aplica a las coordenadas) — la conversión a píxeles de la imagen original es `ancho_px = ancho_linea_prom * (dpi / 72)`, con `dpi` como campo top-level del JSON exportado.

## Celda 1 — Subir los pares PNG + JSON exportados por el portal

In [ ]:
from google.colab import files
import os

os.makedirs('/content/subidos', exist_ok=True)
print('Selecciona los pares .png + .json exportados por el boton "Exportar dataset de muros" del portal (podes subir varios pares juntos):')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'/content/subidos/{name}', 'wb') as f:
        f.write(data)

# Empareja por nombre base (mismo nombre, distinta extension) -- asi los exporta el portal.
bases = sorted({os.path.splitext(n)[0] for n in uploaded})
pares = []
for base in bases:
    png_path = f'/content/subidos/{base}.png'
    json_path = f'/content/subidos/{base}.json'
    if os.path.isfile(png_path) and os.path.isfile(json_path):
        pares.append((png_path, json_path))
    else:
        print(f'AVISO: {base} no tiene el par completo (png+json), se omite')
print(f'{len(pares)} par(es) completo(s) listos para rasterizar')

## Celda 2 — Función de rasterización: (PNG, JSON de muros) → (imagen 256×256, máscara 256×256)

In [ ]:
import cv2
import json
import numpy as np

def rasterizar_par(png_path, json_path):
    """
    Devuelve (img_256, mask_256, meta) a partir del PNG limpio original y el JSON reducido
    exportado por el portal (entry_idx, imagen_w_px, imagen_h_px, mpp, dpi, muros_geo).
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)

    img_bgr = cv2.imread(png_path)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    # Resize directo, sin preservar aspect ratio -- mismo criterio ya verificado en
    # Test_MLSTRUCTFP_2026-08-06.ipynb (asi se le da la imagen al modelo en inferencia,
    # el target de entrenamiento tiene que sufrir la misma distorsion).
    img_256 = cv2.resize(img_gray, (256, 256))

    imagen_w_px = meta['imagen_w_px']
    imagen_h_px = meta['imagen_h_px']
    dpi = meta.get('dpi') or 72
    sx = 256 / imagen_w_px
    sy = 256 / imagen_h_px
    zoom = dpi / 72  # ancho_linea_prom esta en puntos PDF, no en pixeles ni metros

    mask = np.zeros((256, 256), dtype=np.uint8)
    for muro in meta.get('muros_geo', []):
        ancho_linea_prom = muro.get('ancho_linea_prom') or 0
        ancho_px_orig = ancho_linea_prom * zoom
        thickness = max(2, round(ancho_px_orig * (sx + sy) / 2))
        for seg in muro.get('segmentos', []):
            p1 = seg['p1']; p2 = seg['p2']
            x1, y1 = p1[0] if isinstance(p1, list) else p1['x'], p1[1] if isinstance(p1, list) else p1['y']
            x2, y2 = p2[0] if isinstance(p2, list) else p2['x'], p2[1] if isinstance(p2, list) else p2['y']
            p1_256 = (round(x1 * sx), round(y1 * sy))
            p2_256 = (round(x2 * sx), round(y2 * sy))
            cv2.line(mask, p1_256, p2_256, 255, thickness)

    return img_256, mask, meta

## Celda 3 — Procesar todos los pares subidos y armar la estructura de carpetas del dataset

In [ ]:
import csv
import shutil
import datetime

DATASET_ROOT = '/content/dataset/Fase 2/Dataset_Fine_Tuning_MLSTRUCT-FP'
os.makedirs(DATASET_ROOT, exist_ok=True)
manifest_path = os.path.join(DATASET_ROOT, 'dataset_manifest.csv')
manifest_existe = os.path.isfile(manifest_path)

resultados = []
with open(manifest_path, 'a', newline='', encoding='utf-8') as fcsv:
    writer = csv.writer(fcsv)
    if not manifest_existe:
        writer.writerow(['proyecto_nivel', 'entry_idx', 'fname_tag', 'n_muros', 'fecha_exportacion', 'imagen_w_px', 'imagen_h_px'])
    for png_path, json_path in pares:
        base = os.path.splitext(os.path.basename(png_path))[0]
        img_256, mask, meta = rasterizar_par(png_path, json_path)

        proyecto = meta.get('proyecto') or 'proyecto'
        fname_tag = meta.get('fname_tag') or f"pagina{meta.get('pagina', '')}"
        carpeta = os.path.join(DATASET_ROOT, f'{proyecto}_{fname_tag}')
        os.makedirs(os.path.join(carpeta, 'original'), exist_ok=True)
        os.makedirs(os.path.join(carpeta, '256'), exist_ok=True)

        shutil.copy(png_path, os.path.join(carpeta, 'original', f'{base}.png'))
        shutil.copy(json_path, os.path.join(carpeta, 'original', f'{base}.json'))
        cv2.imwrite(os.path.join(carpeta, '256', f'{base}_imagen_256.png'), img_256)
        cv2.imwrite(os.path.join(carpeta, '256', f'{base}_mascara_muro_256.png'), mask)

        n_muros = len(meta.get('muros_geo', []))
        writer.writerow([f'{proyecto}_{fname_tag}', meta.get('entry_idx'), fname_tag, n_muros,
                          datetime.datetime.now().isoformat(timespec='seconds'),
                          meta.get('imagen_w_px'), meta.get('imagen_h_px')])
        resultados.append({'base': base, 'carpeta': carpeta, 'img_256': img_256, 'mask': mask, 'n_muros': n_muros})
        print(f'{base}: {n_muros} muro(s) rasterizado(s) -> {carpeta}')

print(f'\n{len(resultados)} par(es) procesado(s). Manifest: {manifest_path}')

## Celda 4 — Verificación visual (overlay máscara en rojo sobre la imagen)

Confirmar de un vistazo que el rasterizado calzó con los muros reales antes de dar el par por bueno -- si el grosor sale absurdamente grueso o las líneas no calzan de posición, revisar la conversión `zoom`/escala antes de confiar en el resultado.

In [ ]:
import matplotlib.pyplot as plt

for r in resultados:
    overlay = cv2.cvtColor(r['img_256'], cv2.COLOR_GRAY2RGB).astype(float)
    overlay[r['mask'] > 0] = [255, 60, 60]
    plt.figure(figsize=(6, 6))
    plt.imshow(overlay.astype(np.uint8))
    plt.title(f"{r['base']} -- {r['n_muros']} muro(s)")
    plt.axis('off')
    plt.show()

## Celda 5 — Empaquetar y descargar

Descarga un `.zip` con la estructura completa de `Fase 2/Dataset_Fine_Tuning_MLSTRUCT-FP/` (incluye el manifest acumulado) -- descomprimir directo en esa carpeta del repo local, mergeando con lo ya acumulado de corridas anteriores.

In [ ]:
import shutil
from google.colab import files

zip_base = '/content/dataset_muros_export'
shutil.make_archive(zip_base, 'zip', '/content/dataset')
files.download(zip_base + '.zip')